# C5_G2_CHB — SzScan file engine against F2-03's L3 rows (gate G2; Track C, chat C5c)

**What:** the 8 test patients and chb24 (253 recordings, about 300 h): each recording goes through the review application's own path (Phase A of
`web_demo/backend/pipeline_demo.py`, then `engine_file.run_file`, the "each file on its own" mode SzScan 1.0) and its
events are scored with L3's scorer; **gate G2** = every recording equal to F2-03's committed L3 row (B-MCD-F, seed 42:
events, tp, fp, seizures, short, windows; q and top within 1e-9). Raw EDF verified against PhysioNet in the session
(the L3 inventory). A smoke recording is compared before the build (STOP on a difference).
**Account:** one with Internet (chosen by the author). **Accelerator:** None (CPU). **Internet:** ON (pip, git clone,
PhysioNet checksums).
**Input (Add Input):** `combined-siena-scalp-and-chb-mit-eeg-datasets`.
**Output:** `/kaggle/working/c5g2/c5g2_chb` (→ `results/c5_szscan/step2/g2/c5g2_chb`; small files only).
**Runtime:** ≈ 2–4 h (computed: the laptop's measured Phase A + engine rate x 1.45 on 4 workers; PROJ measures it on a 1-h recording and stops above 10.5 h).
Code: `github.com/Nhanvas/Thesis` ref `main`; the sha256 of every module and committed file used is asserted below.
Do not edit; *Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository (read only)
import os, sys, subprocess, time, json, hashlib
T_NB = time.time()
NOTEBOOK, REPO_REF = "C5_G2_CHB", "main"
WORK, INPUT, SCRATCH = "/kaggle/working", "/kaggle/input", "/tmp/r2"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mne==1.12.1", "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
REPO = "/tmp/Thesis"
if not os.path.exists(REPO + "/src/c5_g2.py"):
    subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                          f"&& cd {REPO} && git checkout -q {REPO_REF}", shell=True)
HEAD = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
print("repository", REPO, HEAD, f"({time.time()-T_NB:.0f}s)")

In [ ]:
# START: sha256 of every module and committed file used (the reviewed code only), environment, imports
WANT = {"src/m2l_core.py": "9690cac7f91f52b2", "src/m2l_run.py": "f0f8c563e5c3726a", "src/m2l_sched.py": "ec2841629bdeb87c", "src/eur_abl.py": "0fd303e9b2f3671f", "src/r2_r1.py": "76cbe51ff0180d4a", "src/r2_core.py": "902d15ad4feb3e8a", "src/e16_recording_baseline.py": "f170e1c59f5b0447", "src/e17b_latent_family.py": "3b4f034dca00eaf3", "src/p3c_freeze.py": "dabe4139f6005fd2", "src/p3_levers.py": "feb247d2d08c52ba", "src/p2_scalars.py": "11feeaf3ba95ced4", "src/e22_test_oneshot.py": "669350bbfaaf0f4e", "src/e23_test_look7.py": "88587668baf12a64", "src/x1_external.py": "9962e3e7fe4ddc35", "src/p2_dump_seeds23.py": "0bc887542989310f", "src/e18_cpd_localizer.py": "e18c0ffe37f78e77", "src/e10_kernel_cpd.py": "5c692e33d92fd0fb", "src/dataprep/preprocessing.py": "0172fbea61b28540", "src/continuous_eval.py": "8fea38dabee5ad25", "src/c1b_build_trainset.py": "b4596a8aa3797653", "src/evaluation_protocol.py": "7caacac9b31bd0f7", "src/c5_g2.py": "1b768136991caac6", "web_demo/backend/pipeline_demo.py": "0ec11ba2e743d1d8", "web_demo/backend/engine_file.py": "c9cda0e1f977930a", "web_demo/backend/edf_input.py": "74d25f60c2058bbf", "results/m2l/l3eval/per_file.csv": "0e62a1e675fca030", "results/m2l/l2/selection.json": "70539f67a1069202", "results/m2l/l3short/minlen.json": "1c6468b1d04f713f", "data/models_v2/gae_v2_clean_seed42.pt": "6e310307b046407a", "results/p3c_freeze/freeze.json": "995f44e6dfd25cec"}
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
bad = {n: sha(f"{REPO}/{n}") for n in WANT if sha(f"{REPO}/{n}") != WANT[n]}
assert not bad, f"files differ from the reviewed ones (rebuild the notebook or pin the right commit): {bad}"
os.environ.update(THESIS_REPO=REPO, R2_WORK=WORK, R2_INPUT=INPUT, R2_SCRATCH=SCRATCH, OMP_NUM_THREADS="1",
                  OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1", C5G2_SET="chb")
sys.path[:0] = [REPO + "/src"]
import c5_g2 as G
assert G.__file__ == f"{REPO}/src/c5_g2.py", G.__file__
print("CPU cores:", os.cpu_count(), "| set", os.environ["C5G2_SET"])

In [ ]:
# RUN: inventory (PhysioNet), smoke + PROJ, every recording through the application, gate G2
ERR = None
try:
    G.main(NOTEBOOK, HEAD, workers=int(os.environ.get("C5G2_WORKERS", "0")) or None)
except SystemExit as e:
    ERR = str(e)

In [ ]:
# END: gates, outputs, measured runtime
OUT = os.path.join(WORK, "c5g2", "c5g2_chb")
for f in ("gates.json", "projection.json", "smoke.json", "SUMMARY.md"):
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        print("=====", f); print(open(p).read()[:4000])
for root, _, fs in os.walk(OUT):
    for f in sorted(fs):
        p = os.path.join(root, f); print(os.path.relpath(p, WORK), os.path.getsize(p))
print(f"MEASURED notebook runtime: {(time.time() - T_NB) / 3600:.3f} h")
if ERR:
    raise RuntimeError(f"STOPPED: {ERR}")